# Max-plus eigenstructure of the *C. elegans* connectomes

This notebook reproduces the max-plus analyses used in the thesis for the five connectivity layers: chemical synapses, gap junctions, monoaminergic signalling, neuropeptidergic signalling, and functional signal propagation.

The reusable mathematical implementation remains in `src/`. Dataset-specific loading and harmonisation are isolated in `src/celegans_helpers.jl`.

All matrices follow the convention `A[target, source]`, so the network edge `source → target` is stored at `A[target, source]`.


## 1. Project setup

Before running the notebook for the first time, instantiate the repository environment with `julia --project=. scripts/setup.jl`.

The original datasets are **not** committed to GitHub. Place them under `data/connectomes/` using the filenames documented in the README.


In [3]:
using Pkg
Pkg.instantiate()
Pkg.precompile()

    Updating registry at `~/.julia/registries/General.toml`
   Installed ColorTypes ─ v0.12.3
   Installed Colors ───── v0.13.2
   Installed XLSX ─────── v0.12.4
    Updating `~/Documents/GitHub/maxplus_eigenvectors_28sept/Project.toml`
⌅ [336ed68f] + CSV v0.10.17
  [a93c6f00] + DataFrames v1.8.2
  [86223c79] + Graphs v1.15.0
  [41177cfe] + MaxPlus v0.5.0
  [fdbf4ff8] + XLSX v0.12.4
  [37e2e46d] ~ LinearAlgebra ⇒ v1.12.0
  [2f01184e] ~ SparseArrays ⇒ v1.12.0
    Updating `~/Documents/GitHub/maxplus_eigenvectors_28sept/Manifest.toml`
  [dce04be8] + ArgCheck v2.5.0
  [ec485272] + ArnoldiMethod v0.4.0
⌅ [336ed68f] + CSV v0.10.17
  [6309b1aa] + CodecInflate64 v0.1.3
  [944b1d66] + CodecZlib v0.7.9
  [3da002f7] + ColorTypes v0.12.3
  [5ae59095] + Colors v0.13.2
  [34da2185] + Compat v4.18.1
  [a8cc5b0e] + Crayons v4.2.0
  [9a962f9c] + DataAPI v1.16.0
  [a93c6f00] + DataFrames v1.8.2
  [864edb3b] + DataStructures v0.19.6
  [e2d170a0] + DataValueInterfaces v1.0.0
  [48062228] + FilePathsBase 

In [4]:
using Pkg

repo_root = if isfile(joinpath(pwd(), "Project.toml"))
    pwd()
elseif isfile(joinpath(pwd(), "..", "Project.toml"))
    normpath(joinpath(pwd(), ".."))
else
    error("Run this notebook from the repository root or notebooks/ directory.")
end

Pkg.activate(repo_root)

using Maxplus_eigenvectors
using DataFrames
using SparseArrays

include(joinpath(repo_root, "src", "celegans_helpers.jl"))
using .CElegansHelpers

configure_tropical_display!()

results_dir = joinpath(repo_root, "results", "celegans_thesis")
figures_dir = joinpath(repo_root, "figures", "celegans_thesis")
mkpath(results_dir)
mkpath(figures_dir)

println("Repository: ", repo_root)


  Activating 

I will show (max,+) -Inf and (min,+) +Inf as .Repository: 

project at `~/Documents/GitHub/maxplus_eigenvectors_28sept`


/Users/emmareynerfuentes/Documents/GitHub/maxplus_eigenvectors_28sept/


## 2. Load and harmonise the five network layers

`CElegansHelpers.load_celegans_thesis_layers` reproduces the data harmonisation used in the thesis. The structural datasets are aligned to the same 302-neuron ordering and transposed where required so that all returned matrices already satisfy `A[target, source]`. The functional matrix retains its original 188-neuron ordering.


In [5]:
connectome_dir = joinpath(repo_root, "data", "connectomes")
celegans = load_celegans_thesis_layers(connectome_dir)

layer_order = [
    :synaptic,
    :gap_junction,
    :monoamine,
    :neuropeptide,
    :functional,
]

layer_inputs = celegans.layers

summary = network_summary((
    synaptic=layer_inputs[:synaptic].matrix,
    gap_junction=layer_inputs[:gap_junction].matrix,
    monoamine=layer_inputs[:monoamine].matrix,
    neuropeptide=layer_inputs[:neuropeptide].matrix,
    functional=layer_inputs[:functional].matrix,
))

display(summary)


Row,Layer,Nodes,Edges,Symmetric,Density
,String,Int64,Int64,Bool,Float64
1,synaptic,302,3709,false,0.0406671
2,gap_junction,302,2196,true,0.0240779
3,monoamine,302,2881,false,0.0315885
4,neuropeptide,302,31417,false,0.34447
5,functional,188,1342,false,0.0379697


## 3. Explicit conversion to Max-plus

The source connectomes are ordinary numeric adjacency matrices `C`. They are converted explicitly to max-plus matrices `A` before the eigendecomposition. Numeric zero represents an absent connection in the thesis datasets and is therefore mapped to max-plus bottom, `−Inf`.


In [6]:
maxplus_inputs = Dict{Symbol,Any}()

for layer in layer_order
    C = layer_inputs[layer].matrix
    A = to_maxplus(C; zero_is_absent=true)
    maxplus_inputs[layer] = A

    println(rpad(String(layer), 16),
        " C eltype = ", eltype(C),
        "   A eltype = ", eltype(A))
end


synaptic         C eltype = Float64   A eltype = MP{Float64}
gap_junction     C eltype = Float64   A eltype = MP{Float64}
monoamine        C eltype = Float64   A eltype = MP{Float64}
neuropeptide     C eltype = Float64   A eltype = MP{Float64}
functional       C eltype = Float64   A eltype = MP{Float64}


## 4. Compute and validate the complete eigensystems

For every layer, the complete right eigensystem is computed. Left eigendirections are also computed except where symmetry allows reuse of the right eigensystem. Every returned eigenpair is validated by the corresponding max-plus eigenvector equation.


In [7]:
analyses = Dict{Symbol,Any}()

for layer in layer_order
    println("\n==================== ", uppercase(String(layer)), " ====================")

    analyses[layer] = analyse_network(
        maxplus_inputs[layer];
        labels=layer_inputs[layer].labels,
        name=String(layer),
        validate=true,
        verbose=true,
    )
end



==================== SYNAPTIC ====================
ufnf₀: using Karp maximum cycle mean.
  λ = 37
  plustimes(λ) = 37.0
  number of representative critical nodes = 2
ufnf₀: using Karp maximum cycle mean.
  λ = 15
  plustimes(λ) = 15.0
  number of representative critical nodes = 1
ufnf₀: using Karp maximum cycle mean.
  λ = 0
  plustimes(λ) = 0.0
  number of representative critical nodes = 2
ufnf₀: using Karp maximum cycle mean.
  λ = -22
  plustimes(λ) = -22.0
  number of representative critical nodes = 1
ufnf₀: using Karp maximum cycle mean.
  λ = 22
  plustimes(λ) = 22.0
  number of representative critical nodes = 2
ufnf₀: using Karp maximum cycle mean.
  λ = 0
  plustimes(λ) = 0.0
  number of representative critical nodes = 1
ufnf₀: using Karp maximum cycle mean.
  λ = 15
  plustimes(λ) = 15.0
  number of representative critical nodes = 1
ufnf₀: using Karp maximum cycle mean.
  λ = 37
  plustimes(λ) = 37.0
  number of representative critical nodes = 2
ufnf₀: using Karp maximum cycl

## 5. Global spectral and SCC overview


In [8]:
spectral_tables = DataFrame[]
scc_rows = DataFrame(
    Layer=String[],
    Nodes=Int[],
    SCCs=Int[],
    NonSingletonSCCs=Int[],
    LargestSCC=Int[],
)

for layer in layer_order
    analysis = analyses[layer]

    right = finite_right_spectrum_summary(analysis.right)
    right[!, :Side] .= "right"
    right[!, :Layer] .= String(layer)
    push!(spectral_tables, right)

    if analysis.left !== nothing && !analysis.symmetric
        left = finite_right_spectrum_summary(analysis.left)
        left[!, :Side] .= "left"
        left[!, :Layer] .= String(layer)
        push!(spectral_tables, left)
    end

    stats = scc_statistics(layer_inputs[layer].matrix)
    push!(scc_rows, (
        String(layer),
        size(layer_inputs[layer].matrix, 1),
        stats.count,
        stats.non_singleton,
        stats.largest,
    ))
end

spectral_summary = vcat(spectral_tables...; cols=:union)

display(scc_rows)
display(select(spectral_summary, :Layer, :Side, :Block, :Direction, :Lambda, :Multiplicity, :SupportSize))


Row,Layer,Nodes,SCCs,NonSingletonSCCs,LargestSCC
,String,Int64,Int64,Int64,Int64
1,synaptic,302,11,2,275
2,gap_junction,302,6,2,296
3,monoamine,302,286,1,17
4,neuropeptide,302,17,1,286
5,functional,188,42,1,147


Row,Layer,Side,Block,Direction,Lambda,Multiplicity,SupportSize
,String,String,Int64,Int64,Float64,Int64,Int64
1,synaptic,right,2,1,37.0,2,280
2,synaptic,right,2,2,15.0,1,300
3,synaptic,left,2,1,15.0,1,18
4,synaptic,left,2,2,37.0,2,295
5,gap_junction,right,2,1,1.0,2,2
6,gap_junction,right,3,1,401.0,3,296
7,monoamine,right,2,1,2.5,4,284
8,monoamine,left,2,1,2.5,4,19
9,neuropeptide,right,2,1,14.0,1,290


## 6. Critical classes

The table below reconstructs the critical graph for each finite right projective eigendirection and reports the corresponding critical classes and representative cycles.


In [9]:
critical_tables = Dict{Symbol,DataFrame}()

for layer in layer_order
    critical_tables[layer] = critical_classes_table(analyses[layer])
    println("\n", uppercase(String(layer)))
    display(critical_tables[layer])
end



SYNAPTIC

GAP_JUNCTION

MONOAMINE

NEUROPEPTIDE

FUNCTIONAL


Row,Block,Direction,Lambda,CriticalClass,CriticalLabels,RepresentativeCycle,CycleMean
,Int64,Int64,Float64,Int64,String,String,Float64
1,2,1,37.0,1,"RIAR, SMDVL",RIAR → SMDVL → RIAR,37.0
2,2,2,15.0,1,M4,M4 → M4,15.0


Row,Block,Direction,Lambda,CriticalClass,CriticalLabels,RepresentativeCycle,CycleMean
,Int64,Int64,Float64,Int64,String,String,Float64
1,2,1,1.0,1,"IL2DL, URADL",IL2DL → URADL → IL2DL,1.0
2,3,1,401.0,1,"ALA, CANL, CANR",ALA → CANL → ALA,401.0


Row,Block,Direction,Lambda,CriticalClass,CriticalLabels,RepresentativeCycle,CycleMean
,Int64,Int64,Float64,Int64,String,String,Float64
1,2,1,2.5,1,"NSML, NSMR, RIML, RIMR",NSML → RIML → NSML,2.5


Row,Block,Direction,Lambda,CriticalClass,CriticalLabels,RepresentativeCycle,CycleMean
,Int64,Int64,Float64,Int64,String,String,Float64
1,2,1,14.0,1,PQR,PQR → PQR,14.0


Row,Block,Direction,Lambda,CriticalClass,CriticalLabels,RepresentativeCycle,CycleMean
,Int64,Int64,Float64,Int64,String,String,Float64
1,2,1,0.921569,1,"RIH, SIBVR",RIH → SIBVR → RIH,0.921569


## 7. Layer-specific thesis views

These compact cells expose the recurrent organisation used in the thesis discussion without regenerating every exploratory table or figure.


In [10]:
# Chemical synapses: 11 SCCs, including the large somatic and pharyngeal recurrent components.
syn_stats = scc_statistics(layer_inputs[:synaptic].matrix)
syn_sizes = syn_stats.sizes

println("Synaptic SCC sizes: ", syn_sizes)
display(critical_tables[:synaptic])


Synaptic SCC sizes: [275, 18, 1, 1, 1, 1, 1, 1, 1, 1, 1]


Row,Block,Direction,Lambda,CriticalClass,CriticalLabels,RepresentativeCycle,CycleMean
,Int64,Int64,Float64,Int64,String,String,Float64
1,2,1,37.0,1,"RIAR, SMDVL",RIAR → SMDVL → RIAR,37.0
2,2,2,15.0,1,M4,M4 → M4,15.0


In [11]:
# Gap junctions: dominant electrical critical structure.
gap_stats = scc_statistics(layer_inputs[:gap_junction].matrix)
println("Gap-junction SCC sizes: ", gap_stats.sizes)
display(critical_tables[:gap_junction])


Row,Block,Direction,Lambda,CriticalClass,CriticalLabels,RepresentativeCycle,CycleMean
,Int64,Int64,Float64,Int64,String,String,Float64
1,2,1,1.0,1,"IL2DL, URADL",IL2DL → URADL → IL2DL,1.0
2,3,1,401.0,1,"ALA, CANL, CANR",ALA → CANL → ALA,401.0


Gap-junction SCC sizes: [296, 2, 1, 1, 1, 1]


In [12]:
# Monoaminergic layer: one 17-neuron recurrent SCC with a critical NSM–RIM class.
mono_stats = scc_statistics(layer_inputs[:monoamine].matrix)
println("Monoamine largest SCC: ", mono_stats.largest)
display(critical_tables[:monoamine])


Row,Block,Direction,Lambda,CriticalClass,CriticalLabels,RepresentativeCycle,CycleMean
,Int64,Int64,Float64,Int64,String,String,Float64
1,2,1,2.5,1,"NSML, NSMR, RIML, RIMR",NSML → RIML → NSML,2.5


Monoamine largest SCC: 17


In [13]:
# Neuropeptidergic layer: one dominant 286-neuron SCC and the PQR self-loop.
np_stats = scc_statistics(layer_inputs[:neuropeptide].matrix)
println("Neuropeptide largest SCC: ", np_stats.largest)
display(critical_tables[:neuropeptide])


Row,Block,Direction,Lambda,CriticalClass,CriticalLabels,RepresentativeCycle,CycleMean
,Int64,Int64,Float64,Int64,String,String,Float64
1,2,1,14.0,1,PQR,PQR → PQR,14.0


Neuropeptide largest SCC: 286


In [14]:
# Functional propagation: one 147-neuron recurrent SCC and the RIH–SIBVR critical class.
func_stats = scc_statistics(layer_inputs[:functional].matrix)
println("Functional largest SCC: ", func_stats.largest)
display(critical_tables[:functional])


Row,Block,Direction,Lambda,CriticalClass,CriticalLabels,RepresentativeCycle,CycleMean
,Int64,Int64,Float64,Int64,String,String,Float64
1,2,1,0.921569,1,"RIH, SIBVR",RIH → SIBVR → RIH,0.921569


Functional largest SCC: 147


## 8. Gap-junction sensitivity control

The thesis additionally examines the dominant electrical critical class. The following control removes only the ALA↔CANL/R connections and recomputes the spectrum without modifying the original layer.


In [15]:
C_gap = copy(layer_inputs[:gap_junction].matrix)
labels_gap = layer_inputs[:gap_junction].labels

idx = Dict(label => i for (i, label) in enumerate(labels_gap))
iALA = idx["ALA"]

C_no_ALA_CAN = copy(C_gap)
for can in ("CANL", "CANR")
    iCAN = idx[can]
    C_no_ALA_CAN[iALA, iCAN] = 0
    C_no_ALA_CAN[iCAN, iALA] = 0
end
dropzeros!(C_no_ALA_CAN)

A_no_ALA_CAN = to_maxplus(C_no_ALA_CAN; zero_is_absent=true)
gap_control = analyse_network(
    A_no_ALA_CAN;
    labels=labels_gap,
    name="gap junctions without ALA-CAN edges",
)

display(finite_right_spectrum_summary(gap_control.right))
display(critical_classes_table(gap_control))


ufnf₀: using Karp maximum cycle mean.
  λ = 1
  plustimes(λ) = 1.0
  number of representative critical nodes = 2
ufnf₀: using Karp maximum cycle mean.
  λ = 250
  plustimes(λ) = 250.0
  number of representative critical nodes = 3
ufnf₀: using Karp maximum cycle mean.
  λ = 0
  plustimes(λ) = 0.0
  number of representative critical nodes = 2
ufnf₀: using Karp maximum cycle mean.
  λ = 0
  plustimes(λ) = 0.0
  number of representative critical nodes = 3
gap junctions without ALA-CAN edges
  nodes: 302
  symmetric: true
  right spectrum blocks: 3
  left spectrum blocks: 3
  eigenvector checks: passed


Row,Block,Direction,LambdaMP,Lambda,Multiplicity,SupportSize
,Int64,Int64,MP,Float64,Int64,Int64
1,2,1,1,1.0,2,2
2,3,1,250,250.0,3,296


Row,Block,Direction,Lambda,CriticalClass,CriticalLabels,RepresentativeCycle,CycleMean
,Int64,Int64,Float64,Int64,String,String,Float64
1,2,1,1.0,1,"IL2DL, URADL",IL2DL → URADL → IL2DL,1.0
2,3,1,250.0,1,"PVDL, PVDR, ALA",PVDL → ALA → PVDL,250.0


## 9. Export compact tables

Generated results are written under `results/celegans_thesis/`, which is ignored by Git.


In [17]:
using CSV

CSV.write(joinpath(results_dir, "network_summary.csv"), summary)
CSV.write(joinpath(results_dir, "scc_overview.csv"), scc_rows)
CSV.write(joinpath(results_dir, "spectral_summary.csv"), spectral_summary)

for layer in layer_order
    CSV.write(
        joinpath(results_dir, "$(layer)_critical_classes.csv"),
        critical_tables[layer],
    )
end

println("Results written to: ", results_dir)


Results written to: /Users/emmareynerfuentes/Documents/GitHub/maxplus_eigenvectors_28sept/results/celegans_thesis


## 10. Optional Graphviz figures

The package can regenerate SCC-condensation and critical-structure SVGs for any selected eigendirection. Graphviz must be installed separately. Uncomment and adapt the example below only when a figure is required.


In [18]:
# layer = :synaptic
# spectrum = finite_right_spectrum_summary(analyses[layer].right)
# block = Int(spectrum.Block[1])
# direction = Int(spectrum.Direction[1])
#
# visualize_critical_structure(
#     analyses[layer].matrix,
#     analyses[layer].right,
#     analyses[layer].labels;
#     block=block,
#     direction=direction,
#     filename=joinpath(figures_dir, "$(layer)_critical"),
# )
#
# visualize_scc_structure(
#     analyses[layer].matrix,
#     analyses[layer].right,
#     analyses[layer].labels;
#     block=block,
#     direction=direction,
#     filename=joinpath(figures_dir, "$(layer)_scc"),
# )
